# Matching lab: who builds the counterfactual?
**Econ 521 · Friday lab · Fri 6 Nov 2026 · Joel Reyes Mora**

Last Thursday's lecture ran `matchit()` on the LaLonde data and got an estimate. Today we rebuild that estimate from scratch in Python, so the method stops being a black box.

**Running example.** 185 men in the National Supported Work (NSW) job-training program and 429 men from the Panel Study of Income Dynamics (PSID) who did not take part. Outcome: earnings in 1978 (`re78`). This is the same `lalonde.csv` as the lecture's live R script. You met the NSW data in the first lab of the semester. Today the comparison group is *observational*, so selection is back.

**By the end of the lab you should be able to:**
1. Measure selection with a balance table, and say which way it biases the naive comparison.
2. Code exact matching and nearest-neighbour (Mahalanobis) matching yourself.
3. Read any matching estimator as **a set of weights on the comparison group**. That idea carries straight into next week's synthetic control lab.
4. Check balance beyond means, and see how one design choice (the number of neighbours) trades bias against variance.

| Section | Topic | Minutes |
|---|---|---|
| 0 | Setup | 3 |
| 1 | Selection in numbers (TODO 1) | 8 |
| 2 | Exact matching by hand (TODO 2) | 7 |
| 3 | Nearest-neighbour matching from scratch (TODO 3) | 12 |
| 4 | Ties, and matching as weights | 6 |
| 5 | Did matching balance the data? (TODO 4) | 7 |
| 6 | One knob: the number of neighbours | 5 |
| 7 | Wrap-up and exit ticket | 2 |

`# TODO` cells have a `check()` line that prints ✓ or ✗ without giving the answer away. Every cell after a TODO runs even if you skip the TODO.

## 0 · Setup
Put `lalonde.csv` (posted on Canvas) in the same folder as this notebook. You need only `numpy`, `pandas`, `scipy`, `statsmodels` and `matplotlib`; Colab already has them.

In [ ]:
# !pip install numpy pandas scipy statsmodels matplotlib   # uncomment if needed
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy.spatial.distance import cdist

pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
BLUE, ORANGE, GREY = "#2b6cb0", "#dd6b20", "#a0aec0"   # blue = treated, orange = comparison
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

path = Path("lalonde.csv") if Path("lalonde.csv").exists() else Path("data/lalonde.csv")
ld = pd.read_csv(path)
for r in ["black", "hispan", "white"]:
    ld[r] = (ld["race"] == r).astype(int)

COVS = ["age", "educ", "black", "hispan", "married", "nodegree", "re74", "re75"]   # used in the distance
BAL = COVS + ["white"]                                                        # balance tables also show white
BENCHMARK = 1794   # experimental estimate, Dehejia & Wahba (1999)
trt, ctl = ld[ld.treat == 1], ld[ld.treat == 0]
print(f"{len(ld)} people: {len(trt)} NSW participants, {len(ctl)} PSID comparison")
ld.head()

In [ ]:
import hashlib as _hl, json as _json
_ANS = _json.loads('{"smd_white": {"s": 0.0093865, "h": "b18caf15598660c3"}, "smd_re74": {"s": 0.0036054999999999998, "h": "0a5613ecc8d53320"}, "att_exact": {"s": 6.549600000000001, "h": "858719ef1bcedfbb"}, "att_nn_first": {"s": 3.12905, "h": "a724a49224e109b4"}, "smd_re75_after": {"s": 0.0006895, "h": "3b67fbe89d990f42"}, "zero74_matched": {"s": 0.0026755000000000004, "h": "efb4dcdf3186a523"}, "smd_weighted_test": {"s": 0.0017677650000000001, "h": "a75a9b8d30f3d111"}}')

def check(name, value):
    """Prints ✓ or ✗ for a TODO answer. Only hashes are stored, so the answers are not in the notebook."""
    if value is None:
        print(f"[{name}] not answered yet")
        return
    try:
        v = float(value)
    except Exception:
        print(f"[{name}] ✗ expected a single number")
        return
    rec = _ANS[name]
    k = int(round(v / rec["s"]))
    ok = any(_hl.sha256(f"econ521-f26|{name}|{k + d}".encode()).hexdigest()[:16] == rec["h"] for d in (-1, 0, 1))
    print(f"[{name}] {'✓ correct' if ok else '✗ not quite, check your code'}")

## 1 · Selection in numbers (≈ 8 min)

From the lecture, the naive comparison splits into two pieces:

$$\underbrace{E[Y\mid D=1]-E[Y\mid D=0]}_{\text{difference in means}} \;=\; \underbrace{E[Y_1-Y_0\mid D=1]}_{\text{ATT}} \;+\; \underbrace{E[Y_0\mid D=1]-E[Y_0\mid D=0]}_{\text{selection bias}}.$$

The NSW experiment gives us something rare: a credible value for the ATT, **$1,794**. So we can see the selection bias directly.

In [ ]:
naive = smf.ols("re78 ~ treat", data=ld).fit(cov_type="HC1")
print(f"Difference in means: {naive.params['treat']:,.0f}  (robust SE {naive.bse['treat']:,.0f})")
print(f"Experimental benchmark: {BENCHMARK:,}")
print(f"Implied selection bias: {naive.params['treat'] - BENCHMARK:,.0f}")

The naive comparison has the wrong sign. The PSID men would have earned about \$2,400 more than the trainees **even without the program**. Why? A balance table shows how the two groups differ.

**Standardized mean difference (SMD).** As in the lecture, divide the gap in means by the treated group's standard deviation, since the target is the effect on the treated:
$$\text{SMD}_k = \frac{\bar x_{k,T} - \bar x_{k,C}}{s_{k,T}}.$$
A common rule of thumb flags |SMD| > 0.1 as imbalance.

### TODO 1 · Write an SMD function and build the raw balance table

In [ ]:
def smd(x_t, x_c, w_c=None):
    '''Standardized mean difference: (treated mean - comparison mean) / treated SD.
    w_c: optional weights for the comparison units (equal weights if None). The comparison
    mean must be the WEIGHTED mean: we reuse this function after matching.'''
    if w_c is None:
        w_c = np.ones(len(x_c))
    # TODO: replace None with the formula. Hint: np.average(x_c, weights=w_c), np.std(x_t, ddof=1)
    return None

raw_bal = pd.DataFrame({
    "treated mean": trt[BAL].mean(),
    "comparison mean": ctl[BAL].mean(),
    "SMD": [smd(trt[k].to_numpy(), ctl[k].to_numpy()) for k in BAL],
})
raw_bal

In [ ]:
white_smd = smd(trt["white"].to_numpy(), ctl["white"].to_numpy())
re74_smd = smd(trt["re74"].to_numpy(), ctl["re74"].to_numpy())
check("smd_white", white_smd)
check("smd_re74", re74_smd)
check("smd_weighted_test", smd(np.array([1.0, 3.0]), np.array([0.0, 2.0, 4.0]), w_c=np.array([1.0, 1.0, 2.0])))

# Safety net: if TODO 1 is unfinished, a reference smd() takes over so later cells still run.
if smd(np.array([1.0, 3.0]), np.array([0.0, 0.0])) is None:
    print("Using a reference smd() so the rest of the notebook runs. Come back to TODO 1!")
    def smd(x_t, x_c, w_c=None):
        w_c = np.ones(len(x_c)) if w_c is None else w_c
        return (np.mean(x_t) - np.average(x_c, weights=w_c)) / np.std(x_t, ddof=1)

> **Quick think (1 min, with your neighbour).** The PSID men are older, more often white and married, and earned about \$3,500 more in 1974. Each of these predicts *higher* earnings in 1978 with or without training. Which way does that push the naive comparison? Does that match the sign of the selection bias above?

## 2 · Exact matching by hand (≈ 7 min)

Exact matching takes the matching idea literally. Within each cell of identical characteristics, compare the trainees with the PSID men in that cell. Then average the within-cell gaps, weighting each cell by its number of **treated** units (the target is the ATT):

$$\widehat{ATT}_{\text{exact}} = \sum_{c} \frac{N_{1c}}{N_1}\left(\bar Y_{1c} - \bar Y_{0c}\right).$$

We match exactly on three binary characteristics: race × married × no degree. That gives 12 cells.

In [ ]:
CELLS = ["race", "married", "nodegree"]
cell_tab = (ld.groupby(CELLS + ["treat"])["re78"].agg(["mean", "size"])
              .unstack("treat"))
cell_tab.columns = ["mean_C", "mean_T", "n_C", "n_T"]
cell_tab = cell_tab[["n_T", "n_C", "mean_T", "mean_C"]]
cell_tab

### TODO 2 · Compute the exact-matching ATT from `cell_tab`
Drop any cell with no treated or no comparison units (here there are none). Then compute the weighted average of the within-cell gaps.

In [ ]:
ok = cell_tab.dropna()
att_exact = None   # TODO: weighted average of (mean_T - mean_C), weights n_T
print(f"Treated units with an exact match: {ok['n_T'].sum():.0f} of {len(trt)}")
check("att_exact", att_exact)

**The same estimator, written as weights.** Every PSID man in cell $c$ gets weight $N_{1c}/N_{0c}$, and every trainee gets weight 1. The weighted comparison group then has the treated group's mix of cells. The balance check below uses those weights. It covers the matching variables and also the ones left out.

In [ ]:
w_exact = ld.join(ok[["n_T", "n_C"]], on=CELLS)
w_exact = np.where(w_exact["treat"] == 1, 1.0, w_exact["n_T"] / w_exact["n_C"])
wc_exact = w_exact[ld.treat.to_numpy() == 0]

bal_exact = pd.DataFrame({
    "SMD raw": [smd(trt[k], ctl[k]) for k in BAL],
    "SMD exact-matched": [smd(trt[k], ctl[k], wc_exact) for k in BAL],
}, index=BAL)
bal_exact

The three matched variables are balanced exactly (SMD = 0). But education (0.22) and 1974 earnings (−0.17) are still out of balance. Could we just add schooling, age and earnings to the cells? The lecture showed what happens then: only 13 of 185 trainees still have an exact twin. Continuous characteristics need a notion of **close** instead of **identical**.

## 3 · Nearest-neighbour matching from scratch (≈ 12 min)

**Distance.** The Mahalanobis distance between person $i$ and person $j$ is
$$d(i,j) = \sqrt{(x_i - x_j)'\,S^{-1}\,(x_i - x_j)}.$$
Read it as two steps. First, put every characteristic in standard-deviation units, so that a year of age and a dollar of earnings are comparable. Second, undo correlation, so that two characteristics that move together (1974 and 1975 earnings) are not counted twice. For $S$ we use the **pooled within-group covariance matrix** (Rubin 1980).

**Matching rule (as in the lecture):** one neighbour, with replacement, effect on the treated:
$$\widehat{ATT} = \frac{1}{N_1}\sum_{i:D_i=1}\big(Y_i - Y_{j(i)}\big),\qquad j(i) = \arg\min_{j:D_j=0} d(i,j).$$

In [ ]:
X_t = trt[COVS].to_numpy(float)
X_c = ctl[COVS].to_numpy(float)
Y_t = trt["re78"].to_numpy()
Y_c = ctl["re78"].to_numpy()
n_t, n_c = len(X_t), len(X_c)

S = ((n_t - 1) * np.cov(X_t, rowvar=False) + (n_c - 1) * np.cov(X_c, rowvar=False)) / (n_t + n_c - 2)
S_inv = np.linalg.inv(S)

# scipy computes the full 185 x 429 matrix of Mahalanobis distances
Dmat = cdist(X_t, X_c, metric="mahalanobis", VI=S_inv)
print("Distance matrix:", Dmat.shape)
print(f"Correlation of re74 and re75 in the pooled sample: {np.corrcoef(ld.re74, ld.re75)[0, 1]:.2f}")

### TODO 3 · (a) one distance by hand, (b) the matches and the ATT
(a) Compute $d(i,j)$ for the **first trainee** (`X_t[0]`) and the **first PSID man** (`X_c[0]`) with `numpy`. Check that it equals `Dmat[0, 0]`.

(b) For every trainee, find the index of the nearest PSID man (`argmin` along each row of `Dmat`). Then compute the ATT. `np.argmin` returns the **first** minimum it finds; keep that for now.

In [ ]:
# (a)
diff = X_t[0] - X_c[0]
d_00 = None          # TODO: sqrt(diff' S_inv diff)
print("by hand:", d_00, "| scipy:", Dmat[0, 0])

# (b)
match_first = None   # TODO: index of the nearest comparison unit for each treated unit (length 185)
att_nn_first = None  # TODO: mean over treated of Y_t - Y_c[match]
check("att_nn_first", att_nn_first)

**A matching card.** To make this concrete, here is one trainee next to the three PSID men closest to him.

In [ ]:
i = 4   # pick any trainee 0-184
nearest3 = np.argsort(Dmat[i], kind="stable")[:3]
card = pd.concat([trt.iloc[[i]][COVS + ["re78"]].assign(role="trainee", distance=0.0),
                  ctl.iloc[nearest3][COVS + ["re78"]].assign(role="PSID match", distance=Dmat[i, nearest3])])
card.set_index("role")

Matching makes no assumption about how `re78` depends on the covariates. All it assumes is that the PSID man's 1978 earnings stand in for what this trainee would have earned untrained. This is the conditional-independence requirement from the lecture, applied to one person at a time.

## 4 · Why doesn't my number match the R output? Ties, and matching as weights (≈ 6 min)

In the lecture, `matchit()` with the same settings reported **\$416**. You just got **\$626**. The code is not wrong. The PSID file contains men with *identical* covariates, so some trainees have two equally near neighbours, and the two neighbours earned different amounts in 1978.

In [ ]:
dmin = Dmat.min(axis=1, keepdims=True)
tied = np.isclose(Dmat, dmin, rtol=1e-10, atol=1e-12)     # True where a control is a nearest neighbour
n_tied = tied.sum(axis=1)
print(f"Trainees with more than one nearest neighbour: {(n_tied > 1).sum()} of {n_t}")

i = np.flatnonzero(n_tied > 1)[0]
pd.concat([trt.iloc[[i]][COVS + ["re78"]].assign(role="trainee"),
           ctl.iloc[np.flatnonzero(tied[i])][COVS + ["re78"]].assign(role="tied PSID twin")]).set_index("role")

In [ ]:
match_first_ = Dmat.argmin(axis=1)                             # first minimum (what argmin returns)
match_last = n_c - 1 - Dmat[:, ::-1].argmin(axis=1)          # last minimum instead of first
W_ties = tied / n_tied[:, None]                                # split the match equally among tied twins
est = {"first tied twin": np.mean(Y_t - Y_c[match_first_]),
       "last tied twin": np.mean(Y_t - Y_c[match_last]),
       "average over tied twins": np.mean(Y_t - W_ties @ Y_c)}

rng = np.random.default_rng(521)
draws = [np.mean(Y_t - Y_c[[rng.choice(np.flatnonzero(r)) for r in tied]]) for _ in range(1000)]

fig, ax = plt.subplots(figsize=(7, 2.8))
ax.hist(draws, bins=40, color=GREY)
for (lab, v), col in zip(est.items(), ["k", "k", BLUE]):
    ax.axvline(v, color=col, lw=2 if col == BLUE else 1, ls="-" if col == BLUE else ":")
    ax.text(v, ax.get_ylim()[1] * 0.92, f" {lab}\n ${v:,.0f}", fontsize=8, color=col)
ax.axvline(416, color=ORANGE, lw=1.5); ax.text(416, ax.get_ylim()[1] * 0.45, " lecture\n $416", color=ORANGE, fontsize=8)
ax.set_xlabel("1-NN ATT estimate ($) under random tie-breaking, 1,000 draws"); ax.set_yticks([])
plt.tight_layout(); plt.show()
pd.Series(est).round(0)

**Lesson.** For 23 of the 185 trainees, the "nearest neighbour" is a coin flip between two PSID twins. Depending on how those flips land, the estimate can be anywhere from about \$100 to \$700. `MatchIt` broke the ties with its own internal rule, and \$416 is one possible draw. The defensible fix is to **split the match equally among tied twins** (Abadie & Imbens 2006 do this, and so does R's `Matching` package). From here on we use the tie-averaged estimate.

### Matching is a set of weights on the comparison group
Every matching estimator can be written as
$$\widehat{ATT} = \bar Y_T - \sum_{j:D_j=0} w_j\,Y_j,\qquad w_j = \frac{1}{N_1}\sum_{i:D_i=1}\frac{\mathbf 1[j\text{ is a match for } i]}{\#\text{matches of } i}.$$
The weights are non-negative and sum to one. They tell you **who builds the counterfactual**.

In [ ]:
w_nn = W_ties.sum(axis=0) / n_t            # weight of each PSID man; sums to 1
att_nn = Y_t.mean() - w_nn @ Y_c
ess = 1 / np.sum(w_nn**2)                   # Kish effective sample size of the comparison group
print(f"Tie-averaged 1-NN ATT: {att_nn:,.0f}")
print(f"PSID men with positive weight: {(w_nn > 0).sum()} of {n_c}")
print(f"Largest weight: {w_nn.max():.3f}  (this man is the match for {w_nn.max() * n_t:.0f} trainees)")
print(f"Effective number of comparison units: {ess:.1f}")

fig, ax = plt.subplots(figsize=(7, 2.6))
srt = np.sort(w_nn)[::-1]
ax.bar(range(n_c), srt, color=ORANGE, width=1.0)
ax.set_xlabel("PSID men, sorted by weight"); ax.set_ylabel("weight $w_j$")
ax.set_title("1-NN matching: most of the comparison group gets weight zero", fontsize=10)
plt.tight_layout(); plt.show()

Most of the 429 PSID men get weight zero. A few dozen carry the whole counterfactual, and one man counts 18 times. **Keep this picture in mind for next Friday.** Synthetic control also builds a counterfactual as non-negative weights that sum to one, and it too gives zero weight to most comparison units.

## 5 · Did matching balance the data? (≈ 7 min)
Balance is checked with the matching weights: compare the treated means with the **weighted** comparison means.

### TODO 4 · Balance after matching
Use your `smd()` function with `w_c = w_nn` to fill the "after" column. Then plot both columns.

In [ ]:
bal = pd.DataFrame({"before": [smd(trt[k], ctl[k]) for k in BAL]}, index=BAL)
bal["after"] = None   # TODO: list comprehension over BAL using smd(..., w_c=w_nn)
bal_after_re75 = bal.loc["re75", "after"]
check("smd_re75_after", bal_after_re75)
bal

In [ ]:
# Love plot (computed here directly, so it runs even if TODO 4 is blank)
before = np.array([abs(smd(trt[k], ctl[k])) for k in BAL])
after = np.array([abs(smd(trt[k], ctl[k], w_nn)) for k in BAL])
order = np.argsort(before)
fig, ax = plt.subplots(figsize=(6, 3.4))
ax.scatter(before[order], range(len(BAL)), color=GREY, label="before matching", zorder=3)
ax.scatter(after[order], range(len(BAL)), color=BLUE, label="after 1-NN matching", zorder=3)
ax.axvline(0.1, color="k", ls=":", lw=1)
ax.set_yticks(range(len(BAL))); ax.set_yticklabels(np.array(BAL)[order])
ax.set_xlabel("|standardized mean difference|"); ax.legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.show()
print(f"Largest |SMD|: {before.max():.2f} before, {after.max():.2f} after")

The largest gap falls from 1.88 to 0.14, which matches the lecture slide. But a balance table compares **means**. Look at the share of men with **zero** earnings in 1974, a sign of being out of work just before the program.

In [ ]:
zero_t = (trt["re74"] == 0).mean()
zero_c_raw = (ctl["re74"] == 0).mean()
zero_c_matched = None   # TODO: weighted share of zeros among PSID men, weights w_nn
print(f"Zero 1974 earnings: trainees {zero_t:.1%} | PSID raw {zero_c_raw:.1%} | PSID matched {zero_c_matched}")
check("zero74_matched", zero_c_matched)

> **Discuss (2 min).** The SMD on `re74` is only 0.06, yet 71% of trainees but only about 54% of their matches earned nothing in 1974. How can both be true? (Hint: what does the Mahalanobis distance think of \$0 against \$600?) Why might the gap matter for 1978 earnings? Name one way to change the match so that zeros are matched to zeros.

## 6 · One knob: the number of neighbours (≈ 5 min)
Matching each trainee with his $k$ nearest PSID men (ties at the $k$-th distance are included) has two effects:
- **more comparison units**, so a less noisy counterfactual and a smaller standard error;
- **worse matches**, since the 4th-nearest man is further away than the nearest, so more bias.

Run the cell and read the table and the figure.

In [ ]:
def knn_weights(Dmat, k):
    '''Weights on comparison units for k-NN matching with replacement (ties at the k-th distance included).'''
    kth = np.sort(Dmat, axis=1)[:, k - 1:k]
    M = Dmat <= kth + 1e-10
    return (M / M.sum(axis=1, keepdims=True)).sum(axis=0) / Dmat.shape[0], M

def weighted_att_se(w_c):
    '''ATT and robust SE from a weighted regression of re78 on treat (as in the lecture's R code).'''
    used = w_c > 0
    y = np.r_[Y_t, Y_c[used]]
    d = np.r_[np.ones(n_t), np.zeros(used.sum())]
    wt = np.r_[np.ones(n_t), w_c[used] / w_c[used].mean()]
    fit = sm.WLS(y, sm.add_constant(d), weights=wt).fit(cov_type="HC1")
    return fit.params[1], fit.bse[1]

rows = []
for k in [1, 2, 3, 4, 5, 7, 10, 15, 20]:
    w_k, M = knn_weights(Dmat, k)
    att, se = weighted_att_se(w_k)
    rows.append({"k": k, "ATT": att, "SE": se, "PSID used": (w_k > 0).sum(), "eff. N": 1 / np.sum(w_k**2),
                 "mean distance": (Dmat * M).sum() / M.sum(),
                 "max |SMD|": max(abs(smd(trt[c], ctl[c], w_k)) for c in BAL)})
knn = pd.DataFrame(rows).set_index("k")
knn

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
ax = axes[0]
ax.errorbar(knn.index, knn["ATT"], yerr=1.96 * knn["SE"], fmt="o-", color=BLUE, capsize=3)
ax.axhline(BENCHMARK, color="k", ls="--", lw=1); ax.text(20, BENCHMARK, " experiment", va="bottom", ha="right", fontsize=8)
ax.axhline(0, color=GREY, lw=0.8)
ax.set_xlabel("number of neighbours k"); ax.set_ylabel("ATT on 1978 earnings ($)")
ax.set_title("Estimate and 95% CI", fontsize=10); ax.set_xticks(knn.index)
ax = axes[1]
ax.plot(knn.index, knn["max |SMD|"], "o-", color=ORANGE)
ax.axhline(0.1, color="k", ls=":", lw=1)
ax.set_xlabel("number of neighbours k"); ax.set_ylabel("largest |SMD| after matching")
ax.set_title("Balance gets worse as k grows", fontsize=10); ax.set_xticks(knn.index)
plt.tight_layout(); plt.show()

**Reading the figure.**
- As $k$ grows the estimate climbs toward the benchmark while balance gets *worse*. Getting closer to the benchmark here is luck, not evidence that larger $k$ is right. In a real study there is no benchmark.
- The confidence intervals are \$3,200–\$4,000 wide and overlap almost completely. With 185 trainees, differences of a few hundred dollars between specifications are noise.
- **Rule:** choose the design (variables, distance, $k$, ties) from balance and overlap **before** you look at the outcome. Report the choices, as the lecture's last slide asked.

> **Quick think.** The lecture's regression on covariates gave \$1,548, closer to \$1,794 than every matching estimate in the table. Does that make regression the better method for this data?

## 7 · Wrap-up (≈ 2 min)

1. **Selection is measurable.** A balance table shows how the comparison group differs. In LaLonde every major difference pushes the naive estimate down.
2. **Exact matching** is subclassification. It balances the matched variables perfectly and does nothing for the others.
3. **Nearest-neighbour matching** replaces "identical" with "close". The Mahalanobis distance standardizes and de-correlates the characteristics.
4. **Every matching estimator is a set of non-negative weights on the comparison group.** Look at them: how many units carry the counterfactual, how concentrated the weights are, and whether ties are deciding anything.
5. **Balance beyond means.** Check shares (zeros), spreads and overlap, not only SMDs. Make design choices before looking at outcomes.

**Next Friday:** synthetic control. One treated region, and a counterfactual built from non-negative weights on other regions that sum to one. Same idea, different data.

### Exit ticket (on Canvas, before you leave)
1. In one sentence, what does the weight $w_j$ on a PSID man mean in a matching estimate of the ATT?
2. Why can the SMD on `re74` be near zero after matching while the share of zero earners is still far apart?
3. A coauthor picks $k=4$ "because it gets closest to the experimental benchmark." What is wrong with that rule?